# Teen Mental Health — AutoML + MLflow

Notebook de modelado: búsqueda automática del mejor algoritmo e hiperparámetros para clasificar `depression_label` usando **FLAML AutoML**, con seguimiento de experimentos en **MLflow**.

Este entorno corre sobre la imagen Docker definida en `Dockerfile` (dev container), que ya incluye todas las dependencias necesarias.

## 0. Dependencias

Todas las librerías ya están instaladas en la imagen del dev container (ver `Dockerfile`): no se requiere `pip install` en runtime.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
)

from pathlib import Path

from flaml import AutoML
import mlflow

RANDOM_STATE = 42

print('pandas :', pd.__version__)
print('numpy  :', np.__version__)
print('mlflow :', mlflow.__version__)

---
## 1. Carga del dataset preprocesado

In [ ]:
df = pd.read_csv('../data/processed/teen_mental_health_preprocessed.csv')

TARGET = 'depression_label'
X = df.drop(columns=TARGET)
y = df[TARGET]

print(f'Shape: {df.shape}')
print(f'Features ({len(X.columns)}): {list(X.columns)}')
print('\nBalance del target:')
vc = y.value_counts()
print(pd.DataFrame({'n': vc, '%': (100 * vc / len(y)).round(2)}))

---
## 2. División train / test estratificada

Con una clase minoritaria muy pequeña (~2.6 % positivos), el split **estratificado** es obligatorio para que train y test mantengan la misma proporción de clases. Usamos 80/20.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]} muestras')
print(f'  Positivos: {y_train.sum()} ({100 * y_train.mean():.2f}%)')
print(f'Test:  {X_test.shape[0]} muestras')
print(f'  Positivos: {y_test.sum()} ({100 * y_test.mean():.2f}%)')

---
## 3. Tratamiento del desbalance: sample weights

Usamos **sample weights** inversamente proporcionales a la frecuencia de clase (`class_weight='balanced'`). Esto penaliza más los errores sobre la clase minoritaria (depresión) durante el entrenamiento, sin alterar el dataset (a diferencia de SMOTE).

In [ ]:
sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)

w0 = sample_weight[y_train == 0][0]
w1 = sample_weight[y_train == 1][0]
print(f'Peso clase 0 (no depresión): {w0:.4f}')
print(f'Peso clase 1 (depresión):    {w1:.4f}')
print(f'Ratio w1/w0: {w1 / w0:.1f}x  →  cada positivo pesa como {w1 / w0:.1f} negativos')

---
## 4. Configuración de MLflow

Los experimentos se registran localmente en `mlruns/` en la raíz del proyecto. La URI se construye con `Path.resolve().as_uri()` para obtener un `file:///ruta/absoluta` canónico, evitando ambigüedad según el directorio de trabajo desde el que se lance el kernel.

In [ ]:
MLFLOW_TRACKING_URI = 'sqlite:///' + str(Path('../mlflow.db').resolve())
EXPERIMENT_NAME = 'teen-mental-health-automl'

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
exp = mlflow.set_experiment(EXPERIMENT_NAME)

print(f'Experimento:   {exp.name}')
print(f'Tracking URI:  {MLFLOW_TRACKING_URI}')

---
## 5. AutoML con FLAML — objetivo: no dejar pasar ningún caso de depresión

FLAML realiza búsqueda bayesiana sobre el espacio conjunto de algoritmos e hiperparámetros, priorizando configuraciones prometedoras antes de explorar regiones costosas.

**Enfoque tipo screening (análogo a cáncer):** el costo de un falso negativo (no detectar depresión) es mucho mayor que el de un falso positivo (marcar un caso sano para revisión). Esto se traduce en dos decisiones separadas:

1. **Métrica de búsqueda — Average Precision (PR-AUC):** mide qué tan bien el modelo *ordena* los casos positivos por encima de los negativos en todo el rango de umbrales posibles. Un modelo trivial que prediga siempre positivo obtiene AP ≈ prevalencia (~2.6%, pésimo), así que la búsqueda sigue premiando modelos con capacidad real de discriminar — a diferencia de optimizar recall puro sobre una predicción binaria fija, que sí se maximiza trivialmente.
2. **Umbral de decisión (sección 7.3):** una vez elegido el mejor modelo, el umbral de clasificación se calibra aparte para maximizar recall en el holdout (idealmente 100%), reportando el costo en falsos positivos. Ahí es donde realmente se aplica el criterio "atrapar todos los casos, aunque sobren positivos".

**Algoritmos candidatos:**

| Alias | Algoritmo |
|---|---|
| `lgbm` | LightGBM |
| `xgboost` | XGBoost |
| `rf` | Random Forest |
| `extra_tree` | Extra Trees |
| `lrl1` | Logistic Regression (L1) |

In [ ]:
TIME_BUDGET = 180  # segundos
N_SPLITS    = 5

automl = AutoML()

automl_settings = dict(
    task='classification',
    metric='ap',  # average precision (PR-AUC): premia ordenar bien los positivos
    time_budget=TIME_BUDGET,
    eval_method='cv',
    n_splits=N_SPLITS,
    split_type='stratified',
    estimator_list=['lgbm', 'xgboost', 'rf', 'extra_tree', 'lrl1'],
    sample_weight=sample_weight,
    seed=RANDOM_STATE,
    verbose=1,
)

automl.fit(X_train=X_train, y_train=y_train, **automl_settings)

print('Búsqueda AutoML completada.')

---
## 6. Resultados de la búsqueda AutoML

In [ ]:
print('=' * 55)
print(f'  Mejor estimador:      {automl.best_estimator}')
print(f'  Mejor AP (CV val):    {1 - automl.best_loss:.4f}')
print('=' * 55)
print('\nHiperparámetros óptimos:')
for k, v in automl.best_config.items():
    print(f'  {k}: {v}')

In [ ]:
print('Mejor AP de validación por estimador:')
rows = []
for est, loss in automl.best_loss_per_estimator.items():
    if loss is None:
        continue
    rows.append({'estimador': est, 'AP_cv': round(1 - loss, 4)})

per_est = pd.DataFrame(rows).set_index('estimador').sort_values('AP_cv', ascending=False)
per_est

---
## 7. Diagnóstico de overfitting

El resultado de la sección 6 es sospechoso: `rf` y `extra_tree` alcanzan **AP = 1.0000** en CV, y hasta el modelo más simple (`lrl1`, regresión logística L1) llega a **AP = 0.9721** — con una clase positiva de solo **25 casos** en train (5-fold CV ⇒ ~5 positivos por fold de validación) y una prevalencia del 2.6%. Un AP perfecto en ese régimen es señal de alerta, no de éxito.

Diseñamos una batería de tests para distinguir tres explicaciones posibles:

- **(a) Fuga de datos** — duplicados entre train/test, o una variable que por sí sola predice el target.
- **(b) Varianza de muestra pequeña** — con tan pocos positivos, un split "afortunado" puede dar AP=1.0 por puro azar, sin que el modelo generalice.
- **(c) Overfitting real** — el modelo memoriza el train y no generaliza al holdout.

Cada test reconstruye el estimador ganador (`clone(automl.model.estimator)`) y lo reentrena manualmente, de forma independiente al objeto `automl` ya ajustado, para poder repetir el proceso de entrenamiento/validación bajo distintas condiciones.

### 7.1 Fuga de datos: duplicados y solapamiento train/test

Si hay filas duplicadas (o casi duplicadas) que caen a la vez en train y test, el modelo "ve" el test durante el entrenamiento sin que el split lo evite.

In [ ]:
dupes_full = df.duplicated().sum()
dupes_X = X.duplicated().sum()
leak_rows = X_train.merge(X_test, how='inner')

print(f'Filas duplicadas en el dataset completo:         {dupes_full}')
print(f'Filas duplicadas en X (features, sin target):    {dupes_X}')
print(f'Filas de X_test que también aparecen en X_train: {len(leak_rows)}')

if dupes_full > 0 or len(leak_rows) > 0:
    print('\n⚠️  Hay solapamiento de filas entre train y test: posible fuga.')
else:
    print('\n✅  Sin duplicados ni solapamiento train/test.')

### 7.2 Señal univariante: ¿alguna variable separa las clases por sí sola?

Si ninguna variable aislada tiene un AUC alto, la separación casi perfecta observada en CV es multivariante — más compatible con overfitting sobre 14 features sobre solo 25 positivos que con una fuga trivial de una sola columna.

In [ ]:
univariate = []
for col in X.columns:
    auc = roc_auc_score(y_train, X_train[col])
    ap = average_precision_score(y_train, X_train[col])
    univariate.append({'feature': col, 'AUC': auc, 'AP': ap})

univariate_df = pd.DataFrame(univariate).sort_values('AUC', ascending=False).reset_index(drop=True)
print(univariate_df.to_string(index=False))

max_auc = univariate_df['AUC'].max()
print(f'\nMáximo AUC univariante: {max_auc:.4f}')
if max_auc > 0.97:
    print('⚠️  Una sola variable casi separa las clases: revisar posible fuga.')
else:
    print('✅  Ninguna variable aislada explica el AP≈1.0 de la CV — si hay sobreajuste, es multivariante.')

### 7.3 Modelo final: train vs. test

Reentrenamos el mejor estimador (misma clase e hiperparámetros que encontró FLAML) sobre todo `X_train` y lo evaluamos tanto en train como en el holdout `X_test`, nunca visto durante la búsqueda. Un gap grande entre ambos es la firma clásica de overfitting.

In [ ]:
from sklearn.base import clone

final_model = clone(automl.model.estimator)
final_model.fit(X_train, y_train, sample_weight=sample_weight)

def eval_split(model, X_eval, y_eval, label):
    proba = model.predict_proba(X_eval)[:, 1]
    pred = model.predict(X_eval)
    ap = average_precision_score(y_eval, proba)
    auc = roc_auc_score(y_eval, proba)
    f1 = f1_score(y_eval, pred)
    print(f'{label:>6} | AP: {ap:.4f}  ROC-AUC: {auc:.4f}  F1: {f1:.4f}')
    return {'split': label, 'AP': ap, 'ROC_AUC': auc, 'F1': f1}

print(f'Modelo evaluado: {automl.best_estimator}\n' + '-' * 55)
res_train = eval_split(final_model, X_train, y_train, 'TRAIN')
res_test = eval_split(final_model, X_test, y_test, 'TEST')

gap_ap = res_train['AP'] - res_test['AP']
print(f'\nGap AP (train - test): {gap_ap:+.4f}')
if gap_ap > 0.15:
    print('⚠️  Gap grande entre train y test: indicio de overfitting.')
else:
    print('✅  Gap moderado: el modelo generaliza razonablemente al holdout.')

print('\nMatriz de confusión (TEST, umbral 0.5):')
print(confusion_matrix(y_test, final_model.predict(X_test)))

### 7.4 Estabilidad de la CV: repeated stratified k-fold

Con solo ~5 positivos por fold de validación, un único 5-fold CV es una estimación de altísima varianza. Repetimos el 5-fold 10 veces (splits distintos) y miramos la distribución del AP, no un único número.

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold

rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)

cv_aps = []
for tr_idx, val_idx in rskf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    sw_tr = compute_sample_weight(class_weight='balanced', y=y_tr)

    est = clone(automl.model.estimator)
    est.fit(X_tr, y_tr, sample_weight=sw_tr)
    cv_aps.append(average_precision_score(y_val, est.predict_proba(X_val)[:, 1]))

cv_aps = np.array(cv_aps)
print(f'AP — CV repetida (5-fold x 10 = {len(cv_aps)} folds)')
print(f'  media: {cv_aps.mean():.4f}   std: {cv_aps.std():.4f}')
print(f'  min:   {cv_aps.min():.4f}   max: {cv_aps.max():.4f}')
print(f'  folds con AP == 1.0: {(cv_aps == 1.0).sum()} / {len(cv_aps)}')

if cv_aps.std() > 0.05 or cv_aps.min() < 0.7:
    print('\n⚠️  Alta varianza entre folds: el AP=1.0 original probablemente fue un split afortunado, no una señal estable.')
else:
    print('\n✅  El AP se mantiene alto y estable entre folds.')

### 7.5 Test de permutación de etiquetas

Sanity check clásico contra overfitting/fuga: barajamos `y_train` al azar y medimos si el modelo *sigue* logrando un AP mucho mayor al esperado por puro azar. Si el AP real no se distingue de la distribución con etiquetas aleatorias, el modelo está explotando artefactos de la muestra, no una señal real.

In [ ]:
from sklearn.model_selection import StratifiedKFold

def cv_ap_for_labels(y_labels, seed):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    aps = []
    for tr_idx, val_idx in skf.split(X_train, y_labels):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_labels.iloc[tr_idx], y_labels.iloc[val_idx]
        sw_tr = compute_sample_weight(class_weight='balanced', y=y_tr)
        est = clone(automl.model.estimator)
        est.fit(X_tr, y_tr, sample_weight=sw_tr)
        aps.append(average_precision_score(y_val, est.predict_proba(X_val)[:, 1]))
    return np.mean(aps)

N_PERM = 20
rng = np.random.RandomState(RANDOM_STATE)

real_ap = cv_ap_for_labels(y_train, seed=RANDOM_STATE)

perm_aps = []
for i in range(N_PERM):
    y_perm = pd.Series(rng.permutation(y_train.values), index=y_train.index)
    perm_aps.append(cv_ap_for_labels(y_perm, seed=RANDOM_STATE + i + 1))

perm_aps = np.array(perm_aps)
p_value = (np.sum(perm_aps >= real_ap) + 1) / (N_PERM + 1)

print(f'AP con etiquetas reales:                              {real_ap:.4f}')
print(f'AP con etiquetas barajadas ({N_PERM} permutaciones):  media={perm_aps.mean():.4f}  std={perm_aps.std():.4f}  max={perm_aps.max():.4f}')
print(f'p-value empírico:                                     {p_value:.4f}')

if p_value < 0.05:
    print('\n✅  El AP real supera claramente al ruido: hay señal real (no es solo overfitting a ruido).')
else:
    print('\n⚠️  El AP real no se distingue del ruido: el modelo podría estar sobreajustando a artefactos, no a señal real.')

### 7.6 Curva de aprendizaje

Si el gap train/validación se achica a medida que agregamos más datos, el problema principal es tamaño de muestra insuficiente. Si el gap se mantiene ancho incluso usando el 100% del train disponible, el problema es la complejidad del modelo relativa a la señal disponible (overfitting estructural).

In [ ]:
train_fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
skf_lc = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
lc_rows = []

for frac in train_fractions:
    fold_train_aps, fold_val_aps = [], []
    for tr_idx, val_idx in skf_lc.split(X_train, y_train):
        X_tr_full, y_tr_full = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

        if frac < 1.0:
            X_sub, _, y_sub, _ = train_test_split(
                X_tr_full, y_tr_full,
                train_size=frac,
                stratify=y_tr_full,
                random_state=RANDOM_STATE,
            )
        else:
            X_sub, y_sub = X_tr_full, y_tr_full

        sw_sub = compute_sample_weight(class_weight='balanced', y=y_sub)
        est = clone(automl.model.estimator)
        est.fit(X_sub, y_sub, sample_weight=sw_sub)

        fold_train_aps.append(average_precision_score(y_sub, est.predict_proba(X_sub)[:, 1]))
        fold_val_aps.append(average_precision_score(y_val, est.predict_proba(X_val)[:, 1]))

    lc_rows.append({
        'frac_train': frac,
        'n_samples': len(X_sub),
        'AP_train': np.mean(fold_train_aps),
        'AP_val': np.mean(fold_val_aps),
        'gap': np.mean(fold_train_aps) - np.mean(fold_val_aps),
    })

lc_df = pd.DataFrame(lc_rows)
print(lc_df.to_string(index=False))

fig_lc, ax_lc = plt.subplots(figsize=(7, 5))
ax_lc.plot(lc_df['frac_train'], lc_df['AP_train'], marker='o', label='Train AP')
ax_lc.plot(lc_df['frac_train'], lc_df['AP_val'], marker='o', label='Validation AP')
ax_lc.set_xlabel('Fracción de datos de entrenamiento')
ax_lc.set_ylabel('Average Precision')
ax_lc.set_title(f'Curva de aprendizaje — {automl.best_estimator}')
ax_lc.set_ylim(0, 1.05)
ax_lc.legend()
ax_lc.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 7.7 Veredicto

Consolidamos las señales de los tests 7.1–7.6 en un diagnóstico único.

In [ ]:
print('RESUMEN — Diagnóstico de overfitting')
print('=' * 60)
print(f'1) Duplicados en dataset completo:        {dupes_full}')
print(f'2) Filas de test filtradas hacia train:   {len(leak_rows)}')
print(f'3) Máx. AUC univariante:                  {max_auc:.4f}')
print(f'4) Gap AP train-test (modelo final):      {gap_ap:+.4f}')
print(f'5) CV repetida — std entre folds:         {cv_aps.std():.4f}  (min={cv_aps.min():.4f})')
print(f'6) Permutación de etiquetas — p-value:     {p_value:.4f}')
print(f'7) Curva de aprendizaje — gap al 100%:    {lc_df.iloc[-1]["gap"]:+.4f}')
print('=' * 60)

flags = []
if dupes_full > 0 or len(leak_rows) > 0:
    flags.append('fuga por duplicados / solapamiento train-test')
if max_auc > 0.97:
    flags.append('fuga univariante (una sola variable casi separa las clases)')
if gap_ap > 0.15:
    flags.append('gap train/test alto en el modelo final')
if cv_aps.std() > 0.05 or cv_aps.min() < 0.7:
    flags.append('CV inestable: varianza alta por clase minoritaria diminuta')
if p_value >= 0.05:
    flags.append('AP no distinguible del ruido (test de permutación)')

if flags:
    print('⚠️  Hipótesis de overfitting CONFIRMADA. Señales encontradas:')
    for f in flags:
        print(f'   - {f}')
    print('\nRecomendación: no confiar en el AP=1.0 de la sección 6. Con solo 25-31')
    print('positivos totales, considerar: (a) recolectar más casos positivos,')
    print('(b) usar nested CV o repeated CV como métrica de reporte en vez de un')
    print('único 5-fold, (c) simplificar el espacio de búsqueda (regularización')
    print('fuerte, menos features) y (d) tratar cualquier métrica de este dataset')
    print('con máxima cautela antes de usarla para decisiones clínicas.')
else:
    print('✅  No se encontró evidencia de overfitting/fuga en estos tests.')

---
## 8. Registro en MLflow: métricas, gráficas y modelo

Toda la evidencia generada en las secciones 6 y 7 (métricas de validación, diagnóstico de overfitting, curvas de evaluación) vive solo en la salida del notebook. Esta sección la persiste como un **run de MLflow** dentro del experimento configurado en la sección 4:

- **Parámetros**: estimador ganador e hiperparámetros de FLAML.
- **Métricas**: AP/ROC-AUC/F1 en train y test, gap train-test, y las métricas del diagnóstico de overfitting (CV repetida, permutación).
- **Gráficas** (como artefactos PNG): curva ROC, curva Precision-Recall, matriz de confusión y curva de aprendizaje.
- **Tablas**: AP por estimador y la tabla de la curva de aprendizaje.
- **Modelo entrenado**: `final_model` (sección 7.3), con su signature de entrada/salida, para poder cargarlo después con `mlflow.pyfunc.load_model` sin re-ejecutar el notebook.

### 8.1 Gráficas de evaluación (ROC, Precision-Recall, matriz de confusión)

Se generan sobre el holdout `X_test`, usando el `final_model` entrenado en la sección 7.3.

In [ ]:
proba_test = final_model.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, proba_test)
prec, rec, _ = precision_recall_curve(y_test, proba_test)

fig_roc, ax_roc = plt.subplots(figsize=(6, 5))
ax_roc.plot(fpr, tpr, color='steelblue', label=f'ROC (AUC={res_test["ROC_AUC"]:.4f})')
ax_roc.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Azar')
ax_roc.set_xlabel('False Positive Rate')
ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title(f'Curva ROC — {automl.best_estimator} (test)')
ax_roc.legend()
ax_roc.grid(alpha=0.3)
plt.tight_layout()
plt.show()

fig_pr, ax_pr = plt.subplots(figsize=(6, 5))
ax_pr.plot(rec, prec, color='salmon', label=f'PR (AP={res_test["AP"]:.4f})')
ax_pr.axhline(y_test.mean(), linestyle='--', color='gray', label=f'Baseline (prevalencia={y_test.mean():.3f})')
ax_pr.set_xlabel('Recall')
ax_pr.set_ylabel('Precision')
ax_pr.set_title(f'Curva Precision-Recall — {automl.best_estimator} (test)')
ax_pr.legend()
ax_pr.grid(alpha=0.3)
plt.tight_layout()
plt.show()

cm_test = confusion_matrix(y_test, final_model.predict(X_test))
fig_cm, ax_cm = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['No depresión', 'Depresión'],
            yticklabels=['No depresión', 'Depresión'], ax=ax_cm)
ax_cm.set_xlabel('Predicción')
ax_cm.set_ylabel('Real')
ax_cm.set_title(f'Matriz de confusión — {automl.best_estimator} (test, umbral 0.5)')
plt.tight_layout()
plt.show()

### 8.2 Run de MLflow: parámetros, métricas, gráficas, tablas y modelo

Todo se registra en un único run dentro del experimento `teen-mental-health-automl` (sección 4).

In [ ]:
import mlflow.sklearn
from mlflow.models import infer_signature

run_name = f'{automl.best_estimator}_overfitting_diagnostics'

with mlflow.start_run(run_name=run_name) as run:
    # Parámetros del mejor modelo (sección 6)
    mlflow.log_param('best_estimator', automl.best_estimator)
    mlflow.log_params(automl.best_config)

    # Métrica de la búsqueda AutoML (sección 6)
    mlflow.log_metric('cv_ap_automl', 1 - automl.best_loss)

    # Métricas train / test del modelo final (sección 7.3)
    mlflow.log_metric('train_ap', res_train['AP'])
    mlflow.log_metric('train_roc_auc', res_train['ROC_AUC'])
    mlflow.log_metric('train_f1', res_train['F1'])
    mlflow.log_metric('test_ap', res_test['AP'])
    mlflow.log_metric('test_roc_auc', res_test['ROC_AUC'])
    mlflow.log_metric('test_f1', res_test['F1'])
    mlflow.log_metric('train_test_ap_gap', gap_ap)

    # Diagnóstico de overfitting (secciones 7.2, 7.4 y 7.5)
    mlflow.log_metric('univariate_max_auc', max_auc)
    mlflow.log_metric('cv_repeated_ap_mean', cv_aps.mean())
    mlflow.log_metric('cv_repeated_ap_std', cv_aps.std())
    mlflow.log_metric('cv_repeated_ap_min', cv_aps.min())
    mlflow.log_metric('permutation_ap_real', real_ap)
    mlflow.log_metric('permutation_ap_mean_shuffled', perm_aps.mean())
    mlflow.log_metric('permutation_p_value', p_value)

    # Gráficas como artefactos (plots/*.png)
    mlflow.log_figure(fig_roc, 'plots/roc_curve.png')
    mlflow.log_figure(fig_pr, 'plots/precision_recall_curve.png')
    mlflow.log_figure(fig_cm, 'plots/confusion_matrix.png')
    mlflow.log_figure(fig_lc, 'plots/learning_curve.png')

    # Tablas como artefactos (tables/*.json)
    mlflow.log_table(data=per_est.reset_index(), artifact_file='tables/ap_per_estimator.json')
    mlflow.log_table(data=lc_df, artifact_file='tables/learning_curve.json')

    # Modelo entrenado, con signature inferida de train
    signature = infer_signature(X_train, final_model.predict_proba(X_train))
    mlflow.sklearn.log_model(
        sk_model=final_model,
        name='model',
        signature=signature,
        input_example=X_train.head(5),
    )

    print(f'Run registrado: {run.info.run_id}')
    print(f'Artifact URI:   {run.info.artifact_uri}')

### 8.3 Registro del modelo en el Model Registry

Usamos `MlflowClient` para registrar el modelo logueado en el run anterior (`run.info.run_id`) bajo un nombre estable en el **Model Registry**. Así, futuros runs con mejores métricas se registran como nuevas versiones del mismo modelo, en vez de quedar dispersos por `run_id`.

In [ ]:
from mlflow import MlflowClient
from mlflow.exceptions import MlflowException

client = MlflowClient()

MODEL_NAME = 'teen-depression-classifier'
model_uri = f'runs:/{run.info.run_id}/model'

try:
    client.create_registered_model(MODEL_NAME)
    print(f'Registered model creado: {MODEL_NAME}')
except MlflowException:
    print(f'Registered model ya existente: {MODEL_NAME}')

model_version = client.create_model_version(
    name=MODEL_NAME,
    source=model_uri,
    run_id=run.info.run_id,
)

client.set_model_version_tag(MODEL_NAME, model_version.version, 'estimator', automl.best_estimator)
client.set_model_version_tag(MODEL_NAME, model_version.version, 'test_ap', f"{res_test['AP']:.4f}")

print(f'\nModelo registrado: {MODEL_NAME}  (versión {model_version.version})')
print(f'Run de origen:      {run.info.run_id}')
print(f'Source URI:         {model_uri}')